# Master Exploratory Data Analysis (EDA) in Python for Data Science

Exploratory Data Analysis (EDA) is an essential step in the Data Science lifecycle. It involves inspecting, summarizing, visualizing, and identifying patterns, anomalies, correlations, and relationships within a dataset before feeding it into machine learning algorithms.

### Key Steps Covered in This Notebook:
1. **Summary Statistics & Overview**: Shape, column types, non-null counts, and metrics.
2. **Univariate Analysis**: Distribution of individual continuous and categorical features.
3. **Bivariate Analysis**: Examining pairwise feature relationships and group statistics.
4. **Multivariate Analysis & Correlation Analysis**: Heatmaps and multi-variable dependencies.
5. **Anomaly & Outlier Detection**: Visualizing extreme values via Boxplots and IQR metrics.
6. **EDA Summary Function**: Creating a automated summary routine for fast EDA on new datasets.

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [5]:
# Set global visualization style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

print(f"Pandas Version: {pd.__version__}")
print(f"Seaborn Version: {sns.__version__}")

Pandas Version: 2.3.3
Seaborn Version: 0.13.2


--- 
## Section 1: Data Generation & Initial Diagnostics

We generate a synthetic customer/employee Data Science dataset with numerical, categorical, and target variables to run visual EDA.

In [2]:
# Set random seed for reproducibility
np.random.seed(42)
n_samples = 200

data = {
    "Age": np.random.normal(loc=35, scale=8, size=n_samples).astype(int),
    "Experience": np.random.normal(loc=7, scale=4, size=n_samples).astype(int),
    "Salary": np.random.normal(loc=75000, scale=18000, size=n_samples),
    "Department": np.random.choice(["Data Science", "Engineering", "Marketing", "Sales"], size=n_samples, p=[0.3, 0.3, 0.2, 0.2]),
    "Education": np.random.choice(["Bachelor", "Master", "PhD"], size=n_samples, p=[0.5, 0.35, 0.15]),
    "Performance_Score": np.random.uniform(50, 100, size=n_samples)
}

# Introduce a few realistic outliers and missing values
df = pd.DataFrame(data)
df.loc[10:12, "Experience"] = np.nan
df.loc[15, "Salary"] = 220000  # Extreme outlier
df["Experience"] = df["Experience"].clip(lower=0)

print("First 5 Rows:")
display(df.head())

print("\n--- Dataset Structural Information ---")
df.info()

print("\n--- Summary Statistics (Numerical) ---")
display(df.describe())

First 5 Rows:


,Age,Experience,Salary,Department,Education,Performance_Score
0,38,8.0,46300.302142,Data Science,Master,88.992270
1,33,9.0,64211.249587,Marketing,Bachelor,55.349032
2,40,11.0,75094.386595,Marketing,Bachelor,88.051395
3,47,11.0,75845.650688,Marketing,Master,77.063329
4,33,1.0,66898.821513,Marketing,Bachelor,98.149600



--- Dataset Structural Information ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Age                200 non-null    int32  
 1   Experience         197 non-null    float64
 2   Salary             200 non-null    float64
 3   Department         200 non-null    object 
 4   Education          200 non-null    object 
 5   Performance_Score  200 non-null    float64
dtypes: float64(3), int32(1), object(2)
memory usage: 8.7+ KB

--- Summary Statistics (Numerical) ---


,Age,Experience,Salary,Performance_Score
count,200.000000,197.000000,200.000000,200.000000
mean,34.180000,6.873096,74250.573169,75.108609
std,7.438025,3.827817,20657.115753,15.011475
min,14.000000,0.000000,30510.398998,50.488542
25%,29.000000,4.000000,60710.188115,61.326673
50%,34.500000,7.000000,73841.169937,74.927097
75%,38.250000,9.000000,85380.343574,87.947670
max,56.000000,22.000000,220000.000000,99.856225


--- 
## Section 2: Univariate Analysis (Distribution of Single Variables)

Univariate analysis examines one feature at a time to understand its shape, spread, skewness, and frequency.

In [ ]:
# 1. Visualizing Numerical Distributions (KDE & Histogram)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(df["Salary"], kde=True, ax=axes[0], color="skyblue")
axes[0].set_title("Salary Distribution (Histogram + KDE)")

sns.histplot(df["Age"], kde=True, ax=axes[1], color="lightgreen")
axes[1].set_title("Age Distribution (Histogram + KDE)")

plt.tight_layout()
plt.show()

# 2. Visualizing Categorical Distributions (Countplot)
plt.figure(figsize=(7, 4))
sns.countplot(data=df, x="Department", palette="Blues_r", order=df["Department"].value_counts().index)
plt.title("Employee Count per Department")
plt.show()

--- 
## Section 3: Bivariate Analysis (Relationship Between 2 Variables)

Exploring correlations, group comparisons, and relationships between two variables.

In [ ]:
# 1. Numerical vs. Numerical: Scatter Plot with Regression Trendline
plt.figure(figsize=(7, 4))
sns.regplot(data=df, x="Experience", y="Salary", color="teal", scatter_kws={"alpha": 0.6})
plt.title("Experience vs Salary (Scatter Plot + Trendline)")
plt.show()

# 2. Categorical vs. Numerical: Box Plot & Violin Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.boxplot(data=df, x="Department", y="Salary", ax=axes[0], palette="Set2")
axes[0].set_title("Salary by Department (Boxplot)")

sns.violinplot(data=df, x="Education", y="Salary", ax=axes[1], palette="Pastel1")
axes[1].set_title("Salary by Education Level (Violinplot)")

plt.tight_layout()
plt.show()

--- 
## Section 4: Multivariate & Correlation Analysis

Analyzing relationships across multiple numerical features simultaneously using Heatmaps and Pairplots.

In [ ]:
# 1. Correlation Matrix Heatmap
plt.figure(figsize=(7, 5))
numeric_df = df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()

sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Numerical Feature Correlation Matrix")
plt.show()

# 2. Multi-feature Breakdown via GroupBy
multi_agg = df.groupby(["Department", "Education"])["Salary"].mean().unstack()
print("Mean Salary cross-tabulated by Department & Education:")
display(multi_agg)

--- 
## Section 5: Automated Outlier & Anomaly Detection

Detecting outliers using the Interquartile Range (IQR) method and visualizing them.

In [ ]:
def detect_outliers_iqr(dataframe, column):
    """Calculates outlier boundaries using the Interquartile Range (IQR)."""
    q1 = dataframe[column].quantile(0.25)
    q3 = dataframe[column].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    
    outliers = dataframe[(dataframe[column] < lower_bound) | (dataframe[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

salary_outliers, lb, ub = detect_outliers_iqr(df, "Salary")
print(f"Salary IQR Outlier Thresholds: Lower = {lb:.2f}, Upper = {ub:.2f}")
print(f"Detected Outliers Count: {len(salary_outliers)}")
display(salary_outliers[["Age", "Department", "Salary"]])

--- 
## Section 6: Automated Fast-EDA Summary Helper Function

Creating a single reusable EDA diagnostic function to inspect new DataFrames instantly.

In [ ]:
def fast_eda(df):
    """Prints a fast diagnostic report for any Pandas DataFrame."""
    print("=================== FAST EDA REPORT ===================")
    print(f"Shape: {df.shape[0]} Rows, {df.shape[1]} Columns\n")
    
    summary = pd.DataFrame({
        "Data Type": df.dtypes,
        "Missing Count": df.isna().sum(),
        "Missing %": (df.isna().sum() / len(df) * 100).round(2),
        "Unique Values": df.nunique()
    })
    display(summary)
    print("=======================================================")

# Run diagnostic function on df
fast_eda(df)

--- 
## EDA Quick Reference & Cheat Sheet

| EDA Objective | Function / Plot | Key Purpose |
| :--- | :--- | :--- |
| **Structural Diagnostic** | `df.info()`, `df.describe()` | Inspect data types, null counts, summary metrics |
| **Distribution Check** | `sns.histplot(df['col'], kde=True)` | Check for skewness, normality, and spread |
| **Category Counts** | `sns.countplot(data=df, x='cat_col')` | Frequency distribution of categorical features |
| **Bivariate Numerical** | `sns.regplot(x='col1', y='col2')` | Check correlation trend and scatter pattern |
| **Bivariate Categorical** | `sns.boxplot(x='cat', y='num')` | Compare distributions across groups |
| **Feature Correlation** | `sns.heatmap(df.corr(), annot=True)` | Multicollinearity and linear relationship discovery |
| **Outlier Detection** | IQR Formula: `Q3 + 1.5 * IQR` | Identify extreme numerical values |